In [1]:
import math
import importlib
import load_data_from_cache
import load_data_from_cache as cache_loader
import Tansformer_src as ts
from torch.utils.data import DataLoader
import torch

In [2]:
train_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/train")
train_emb_loader = DataLoader(train_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
val_emb_ds = cache_loader.CachedEmbeddingDataset('cache/esm2_t30/val')
val_emb_loader = DataLoader(val_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)
test_emb_ds = cache_loader.CachedEmbeddingDataset("cache/esm2_t30/test")
test_emb_loader = DataLoader(test_emb_ds, batch_size=8, shuffle=True, collate_fn=cache_loader.collate_cached)

In [4]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model = ts.TransformerRSA(
    d_in=640,
    d_model=16,
    n_heads=1,
    n_layers=1,
    dim_ff=32,
    dropout=0.0,
    mlp_hidden=8,
    use_input_layernorm=False,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0)

for epoch in range(1, 3):
    tr = ts.train_one_epoch(model, train_emb_loader, optimizer, device)
    va_loss, va_r = ts.eval_one_epoch(model, val_emb_loader, device, clamp_for_metrics=True)
    print(f"epoch {epoch:02d} | train {tr:.5f} | val {va_loss:.5f} | val pearson {va_r:.4f}")

/Users/amirrade/Desktop/SASA-prediction/notebooks/Tansformer_src.py:85: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


epoch 01 | train 0.03684 | val 0.03446 | val pearson 0.7130


KeyboardInterrupt: 